In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your project-root .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("LangChain Groq model initialized successfully.")

LangChain Groq model initialized successfully.


In [2]:
class ResearchExtraction(BaseModel):
    research_question: str = Field(
        description="The main research question or objective stated in the abstract"
    )
    method: str = Field(
        description="The research method or approach described in the abstract"
    )
    key_finding: str = Field(
        description="The main finding reported in the abstract"
    )


parser = PydanticOutputParser(
    pydantic_object=ResearchExtraction
)

print("Research extraction schema created.")
print(parser.get_format_instructions())

Research extraction schema created.
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"research_question": {"description": "The main research question or objective stated in the abstract", "title": "Research Question", "type": "string"}, "method": {"description": "The research method or approach described in the abstract", "title": "Method", "type": "string"}, "key_finding": {"description": "The main finding reported in the abstract", "title": "Key Finding", "type": "string"}}, "required": ["research_question", "method", "key_finding"]}
```


In [3]:
abstract_text = """
This study investigates whether short daily walking sessions are associated
with improved self-reported mood among university students. Researchers
conducted a four-week observational study in which 120 students recorded
their daily walking duration and mood ratings. Students who reported more
walking generally also reported better mood ratings. Because the study was
observational, it cannot establish that walking caused the mood improvement.
"""

print("RESEARCH ABSTRACT")
print(abstract_text)


RESEARCH ABSTRACT

This study investigates whether short daily walking sessions are associated
with improved self-reported mood among university students. Researchers
conducted a four-week observational study in which 120 students recorded
their daily walking duration and mood ratings. Students who reported more
walking generally also reported better mood ratings. Because the study was
observational, it cannot establish that walking caused the mood improvement.



In [4]:
extraction_prompt = PromptTemplate(
    template="""
Extract the research question, method, and key finding from the abstract.

Rules:
- Use only information stated in the abstract.
- Do not invent numerical results or conclusions.
- Preserve important limitations mentioned in the abstract.
- Return the result using the required structured format.

Research abstract:
{abstract_text}

{format_instructions}
""",
    input_variables=["abstract_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

extraction_chain = extraction_prompt | llm | parser

research_data = extraction_chain.invoke({
    "abstract_text": abstract_text
})

print("STEP 1 — STRUCTURED EXTRACTION")
print(research_data.model_dump_json(indent=2))

STEP 1 — STRUCTURED EXTRACTION
{
  "research_question": "Whether short daily walking sessions are associated with improved self-reported mood among university students.",
  "method": "A four‑week observational study in which 120 university students recorded their daily walking duration and self‑reported mood ratings.",
  "key_finding": "Students who reported more walking generally also reported better mood ratings, but because the study was observational it cannot establish that walking caused the mood improvement."
}


In [5]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a science communicator explaining research to a general audience.

Write a clear, accessible summary using only the supplied structured
research information.

Requirements:
- Explain the research question in everyday language.
- Describe the method simply.
- Explain the key finding without exaggerating it.
- Preserve important limitations.
- Do not introduce facts that are absent from the supplied data.
- Return plain text only."""
    ),
    (
        "human",
        "Structured research information:\n{research_data}"
    )
])

summary_chain = summary_prompt | llm | StrOutputParser()

print("Layperson-summary chain created.")

Layperson-summary chain created.


In [6]:
layperson_summary = summary_chain.invoke({
    "research_data": research_data.model_dump_json(indent=2)
})

if not layperson_summary or not layperson_summary.strip():
    raise ValueError("The model returned an empty summary.")

print("STEP 2 — LAYPERSON SUMMARY")
print("=" * 45)
print(layperson_summary)

STEP 2 — LAYPERSON SUMMARY
**What the researchers wanted to know**  
They asked whether taking short walks each day is linked to a better mood among college students.

**How they looked into it**  
For four weeks, 120 university students kept a simple diary. Each day they wrote down how long they walked and gave a quick rating of how they felt that day.

**What they found**  
Students who logged more walking time tended to give higher mood ratings. In other words, more walking was associated with feeling better.

**What the study can’t tell us**  
Because the study only observed students’ habits without any experimental control, it can’t prove that walking caused the mood improvement. Other factors might explain the relationship.


In [7]:
assert isinstance(research_data, ResearchExtraction)
assert research_data.research_question.strip()
assert research_data.method.strip()
assert research_data.key_finding.strip()

assert isinstance(layperson_summary, str)
assert layperson_summary.strip()

print("Validation passed.")
print("Step 1 returned valid structured research information.")
print("Step 2 returned a non-empty layperson summary.")
print("Step 2 uses the structured extraction, not the original abstract.")

Validation passed.
Step 1 returned valid structured research information.
Step 2 returned a non-empty layperson summary.
Step 2 uses the structured extraction, not the original abstract.


In [8]:
test_abstract = """
This paper examines how spaced practice affects vocabulary learning.
Thirty learners studied new words using either repeated sessions spread
over several days or one longer session. The spaced-practice group
remembered more words on a later test. The study was small and focused
on one vocabulary task.
"""

test_research_data = extraction_chain.invoke({
    "abstract_text": test_abstract
})

test_summary = summary_chain.invoke({
    "research_data": test_research_data.model_dump_json(indent=2)
})

assert isinstance(test_research_data, ResearchExtraction)
assert test_summary and test_summary.strip()

print("SECOND TEST — STRUCTURED EXTRACTION")
print(test_research_data.model_dump_json(indent=2))

print("\nSECOND TEST — LAYPERSON SUMMARY")
print(test_summary)

print("\nSecond test passed.")

SECOND TEST — STRUCTURED EXTRACTION
{
  "research_question": "How does spaced practice affect vocabulary learning?",
  "method": "An experimental study with thirty learners who studied new words either in repeated sessions spread over several days (spaced practice) or in one longer session (massed practice).",
  "key_finding": "The spaced‑practice group remembered more words on a later test, although the study was small and focused on a single vocabulary task."
}

SECOND TEST — LAYPERSON SUMMARY
**What the researchers wanted to know**  
They asked whether spreading out study sessions (spaced practice) helps people learn new vocabulary better than studying everything at once in a single, longer session (massed practice).

**How they tested it**  
Thirty participants were divided into two groups. One group practiced the new words over several days, with short study sessions each day. The other group studied the same words in one longer session. After the study period, all participants to

In [9]:
print("""
Conclusion:
The Research Paper Abstract to Layperson Summary Pipeline has two linked
steps. Step 1 uses PromptTemplate and PydanticOutputParser to extract
the research question, method, and key finding. Step 2 uses
ChatPromptTemplate with a science-communicator persona and StrOutputParser
to explain the structured information in plain language.
""")


Conclusion:
The Research Paper Abstract to Layperson Summary Pipeline has two linked
steps. Step 1 uses PromptTemplate and PydanticOutputParser to extract
the research question, method, and key finding. Step 2 uses
ChatPromptTemplate with a science-communicator persona and StrOutputParser
to explain the structured information in plain language.

